# 04 — RAGBench Results Matrix — Group 16

**Goal:** run our RAG pipeline over the 5 RAGBench domains under several **strategies** (configs), score every answer with the validated judge, and produce the **strategy × domain matrix** — the headline Task-1 deliverable.

**This is where everything comes together.** The pieces built brick-by-brick now run as one chain, once per question:

```
retrieve -> rerank -> repack -> prompt -> generate   (the pipeline answers)
       -> segment -> judge -> TRACe score            (the evaluator grades)
```

**An experiment = a config.** The `configs/*.yaml` files each define one pipeline setup. The runner loops `configs × domains`, writes one CSV row per pair = the matrix. The starter grid is a clean 2×2 that varies one lever at a time:

| | reranker `cross_encoder` | reranker `none` |
|---|---|---|
| **prompt `grounded`** | grounded_rerank | grounded_norerank |
| **prompt `minimal`** | minimal_rerank | minimal_norerank |

- **prompt** grounded vs minimal → the **Adherence** lever (does "answer only from context" curb hallucination?).
- **reranker** cross_encoder vs none → the **Relevance** lever (does re-scoring sharpen context?).

> **Prerequisite:** notebook 03 picked the judge. Set `JUDGE_MODEL` below to that winner.
> Runs on **Colab GPU**. The run is **resumable** — already-done `(config, domain)` pairs are skipped, so a disconnect won't lose finished work.

In [ ]:
# Colab setup: clone the repo + install dependencies (pipeline models + judge).
import os, sys
if not os.path.exists("src"):
    !git clone https://github.com/abhisagar123/CapstoneRAG.git
    os.chdir("CapstoneRAG")
sys.path.insert(0, os.getcwd())

!pip install -q datasets transformers accelerate bitsandbytes sentence-transformers faiss-cpu pyyaml nltk

## (optional) Hugging Face token

Set this if you hit download rate-limits, or if you switch to a **gated** model (Llama-3) which won't download without it. **Qwen2.5 is ungated — you can skip this.**

**Safe setup:** add the token in Colab's **🔑 Secrets** panel (left sidebar) as `HF_TOKEN`, toggle *Notebook access* on. The cell below reads it from there — **never paste the token into a cell** (this repo is public; a committed token is a leaked credential).

In [ ]:
# Load HF_TOKEN from Colab Secrets if present. Optional + safe:
#  - no token set  -> silently continue (fine for ungated models like Qwen2.5)
#  - not on Colab   -> silently continue (the import only exists on Colab)
# Never hardcode the token here: this repo is public.
try:
    from google.colab import userdata
    _tok = userdata.get('HF_TOKEN')
    if _tok:
        import os
        os.environ['HF_TOKEN'] = _tok
        print('HF_TOKEN loaded from Colab Secrets.')
    else:
        print('No HF_TOKEN secret set — continuing unauthenticated (ok for ungated models).')
except Exception:
    print('Not on Colab (or Secrets unavailable) — continuing without HF_TOKEN.')

## 1. Settings — what to run

- `JUDGE_MODEL` — the judge you chose in notebook 03 (Qwen2.5-7B by default).
- `GEN_MODEL` — the **generator** (the model whose answers we are grading). Small for a first pass (Qwen2.5-3B); the configs name it too, but this overrides them so you set it in one place.
- `DOMAINS` — which of the 5 to run. Start with 1–2; add the rest once the flow is proven.
- `N` — examples per (config, domain). Small first: total judge calls = `len(configs) × len(DOMAINS) × N`.

In [ ]:
# --- Edit these, then run top-to-bottom ---
JUDGE_MODEL  = "Qwen/Qwen2.5-7B-Instruct"   # the winner from notebook 03
GEN_MODEL    = "Qwen/Qwen2.5-3B-Instruct"   # the generator under test (small for first pass)
LOAD_IN_4BIT = True                          # True on free T4; False on A100/L4

DOMAINS = ["GenKnowledge", "CustomerSupport"]   # start small; full set is the 5 keys of DOMAIN_CONFIGS
N       = 10                                     # examples per (config, domain)

OUT_CSV = "results/ragbench_matrix.csv"

# Gated models (Llama) need a token — set it in the HF_TOKEN cell above (Colab Secrets).

## 2. Register components + build the judge

`load_*()` registers the heavy components (real embedder / generator / reranker / judge) into the registry. Building the judge downloads + loads it onto the GPU (slow, one-time).

In [ ]:
import src                                  # light components
from src.embeddings import load_embedders
from src.generation import load_generators
from src.reranking import load_rerankers
from src.judge import load_judges
load_embedders(); load_generators(); load_rerankers(); load_judges()

from src.registry import build
judge = build("judge", "hf", {"model": JUDGE_MODEL, "load_in_4bit": LOAD_IN_4BIT,
                              "max_new_tokens": 1536})
print(f"Judge ready: {JUDGE_MODEL}")

## 3. Load the configs and build the (config × domain) grid

Each YAML is loaded, its `domain` is overridden per target domain, and its `generator.model` is set to `GEN_MODEL` (one knob, set above). We also attach the **filename** as `config_name` so every matrix row is traceable to its source file — and we guard against two configs collapsing to the same `config_id` (which would overwrite each other in the matrix).

In [ ]:
import glob, yaml, copy
from src.config import from_dict
from src.runner import config_id

config_files = sorted(glob.glob("configs/*.yaml"))
print("config files:", [os.path.basename(p) for p in config_files])

raw_by_name = {}
for path in config_files:
    with open(path) as f:
        raw_by_name[os.path.basename(path)] = yaml.safe_load(f)

# Build one PipelineConfig per (config_file × domain), overriding domain + generator model.
grid = []   # list of (config_name, PipelineConfig)
for name, raw in raw_by_name.items():
    for dom in DOMAINS:
        d = copy.deepcopy(raw)
        d["domain"] = dom
        d["generator"] = {**d.get("generator", {}), "type": "hf",
                          "model": GEN_MODEL, "load_in_4bit": LOAD_IN_4BIT}
        grid.append((name, from_dict(d, do_validate=True)))

# Collision guard: config_id is built from stage TYPES, so two files differing only
# in a PARAM would collide and overwrite in the CSV. The starter grid differs by type,
# but check anyway so a future param-only ablation fails loudly here, not silently.
from collections import Counter
ids_per_domain = Counter((config_id(cfg), cfg.domain) for _, cfg in grid)
clash = [k for k, c in ids_per_domain.items() if c > 1]
assert not clash, f"config_id collision (would overwrite matrix rows): {clash}"
print(f"built {len(grid)} (config x domain) pipelines, all distinct ✅")

## 4. Run the matrix

`examples_for(cfg)` loads that config's domain (cached so each domain loads once). `run_matrix` runs each pipeline, segments + judges + scores every answer, and writes one row per `(config, domain)` to the CSV — flushing per row so a disconnect is safe.

We pass a small wrapper so each row also records `config_name`.

In [ ]:
import csv
from src.data_loader import load_domain
from src.segmentation import OutputSegmenter, RegexSplitter
from src.runner import run_experiment, FIELDNAMES

SEG = OutputSegmenter(RegexSplitter())

# Cache examples per domain (don't reload for each config of the same domain).
_ex_cache = {}
def examples_for_domain(dom):
    if dom not in _ex_cache:
        _ex_cache[dom] = load_domain(dom, split="test", n=N, seed=42)
    return _ex_cache[dom]

# We call run_experiment per (name, cfg) so we can add the config_name column +
# keep the resume-skip behaviour. (run_matrix doesn't carry config_name, so we
# replicate its resumable write loop here with the extra column.)
COLS = ["config_name"] + FIELDNAMES

done = set()
if os.path.exists(OUT_CSV):
    with open(OUT_CSV, newline="") as f:
        done = {(r["config_name"], r["domain"]) for r in csv.DictReader(f)}

write_header = not os.path.exists(OUT_CSV)
with open(OUT_CSV, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLS)
    if write_header:
        w.writeheader()
    for name, cfg in grid:
        if (name, cfg.domain) in done:
            print(f"skip (done): {name} / {cfg.domain}"); continue
        print(f"running {name} on {cfg.domain} (N={N}) ...", flush=True)
        row = run_experiment(cfg, examples_for_domain(cfg.domain), segmenter=SEG, judge=judge)
        w.writerow({"config_name": name, **row}); f.flush()
        print(f"   -> rel={row['relevance']}  util={row['utilization']}  "
              f"compl={row['completeness']}  adh={row['adherence']}")
print("done.")

## 5. The matrix + a first read

Pivot the long CSV into the **strategy × domain** view, per metric. This is the table that goes in the report (alongside a comparison to the dataset's reference scores).

In [ ]:
import pandas as pd
df = pd.read_csv(OUT_CSV)
print("Raw rows:\n")
print(df[["config_name","domain","n","n_scored","relevance","utilization",
          "completeness","adherence","scoring"]].round(3).to_string(index=False))

print("\n\nStrategy × domain, per metric:")
for metric in ["relevance","utilization","completeness","adherence"]:
    print(f"\n--- {metric} ---")
    piv = df.pivot_table(index="config_name", columns="domain", values=metric)
    print(piv.round(3).to_string())

## 6. How to read it + next steps

**What the matrix shows.** Each cell is a strategy's mean TRACe score on a domain. Read it two ways:
- **Down a column** (fixed domain): which strategy wins *here*? Best-per-domain configs are exactly the per-domain "winning configs" the demo will use.
- **Across a row** (fixed strategy): how does one setup hold up across domains? Big swings = domain sensitivity (expected — legal ≠ finance).

**The two levers, isolated.**
- `grounded` vs `minimal` (same reranker) → the prompt's effect on **Adherence** (and Utilization).
- `cross_encoder` vs `none` (same prompt) → the reranker's effect on **Relevance**.

**Caveats (be honest in the report).**
- **Small `N`** — these are estimates to prove the flow + spot trends, not final numbers. Re-run the chosen winners at higher `N` for the cited table.
- **`corpus_mode: per_example`** — each question is answered against only its own documents (matches the reference setup). The *pooled* per-domain corpus is a separate Phase-2 experiment.
- **Chunking barely moves 8/12 sub-datasets** (they ship pre-segmented; see AI_CONTEXT §13) — so don't over-read chunking effects outside cuad/techqa/expertqa.
- **Compare to reference.** RAGBench ships reference TRACe scores; the report's key analysis is *our pipeline's* scores vs those, per domain — not just our configs against each other.

**Next:** pick the best config per domain, then either (a) scale `N` for final numbers, (b) add a chunking ablation on `cuad`, or (c) move to Task 2 (RGB).